# Computational Theory Assignment Problem (SHA-256)


## Problem 1: Representing SHA-256 Data
Before we go ahead with the implementation of SHA-256, I'll first clarify a few things, mainly how input, output, and intermediate data in Python.

NOTE: the small python example I give for each data representation is just a simplified example for demo's sake, the actual implementation is after that.

### 32-bit words
Unlike C/C++/Rust where there's explicit `uint32_t` typing, Python is dynamically typed and its regular integers are not fixed-width. SHA-256 requires every word to behave as an unsigned 32-bit value, so the actual implementation will use NumPy's `np.uint32` type.

Initially, I wanted to go with an `Int32` class wrapper to make it safer, but decided against it because of a few reasons.
1. Performance overhead - SHA-256 produces a 32-bit result in every single operation, and a custom class would create object allocation overhead in the thousands of operations in the function.
2. Lose out on clean bitwise op syntax - a custom class would need many overloaded operators (i.e. `__add__`, `__xor__`, `__rshift__`, etc.), which adds work and more places where things could go wrong.
3. KISS principle, this one is self explanatory

The small example above uses a native Python `int` with a 32-bit mask to demonstrate the idea. In the actual implementation, the reusable `to_uint32` helper applies the mask and converts the result to `np.uint32`, keeping every SHA-256 word within the required 32-bit range.

In [1]:
# this is just for demo, we'll be encapsulating the masking in a helper function later on
word = 0x12345678
word = word & 0xffffffff

print("word in decimal = ", word) # 305419896 (0x12345678 in decimal)
print("word as hexadecimal = ", f"{word:08x}")  # 12345678
# x means hex, 08 means 8 digit, pad if necessary, same format as in C printf

word in decimal =  305419896
word as hexadecimal =  12345678


### sequences of 32-bit words
For the actual implementation, I'll use a mutable NumPy array with `dtype=np.uint32` to represent a sequence of words. This keeps each element fixed at 32 bits and is convenient when the message schedule is mutated during SHA-256.

The earlier `list[int]` example is a simplified demonstration of a word sequence. A Python list is easy to understand, but it does not enforce the 32-bit unsigned representation. The `to_uint32_array` helper below creates the NumPy representation. `bytes` and `bytearray` are alternatives for raw byte data, but word operations would require repeated conversion between bytes and integers.

In [10]:
words = [0x12345678, 0x90abcdef, 0x00000001] # list of 3 words as a demo

print("len(words) = ", len(words))
print([f"{word:08x}" for word in words])

len(words) =  3
['12345678', '90abcdef', '00000001']


### input messages
For the input messages we'll use python's immutable `bytes` type because sha256 operates at a byte level, the `bytes` type is evidently byte-level precise, I'm not going to be using characters because UTF-8 strings can range from 1-4 bytes long e.g. ඞ (0xE0 0xB6 0x9E) is 3 bytes long meanwhile the char `a` is 1 byte long. 

To give further clarification on python's `bytes` type, under the hood, it's literally just a contiguous array of 1 byte values, but with python magic wrapped over it. TLDR it behaves like `const uint8_t[]`. It's also handy because out of the box it provides slicing that's easy to use. An example of the slicing behaviour is in the code below.

I suppose we could theoretically use `str` or `list` here too but that would require extra work. 

e.g. using `str` forces me to encode to bytes before calculating exact bit lengths or applying masks anyway

e.g.2 using `list[int]` of raw byte values wastes memory, loses the super easy byte slicing provided by `bytes` and also require extra conversion steps during the block processing, which is really unnecessary

In [5]:
message = "abc"
message_bytes = message.encode("utf-8")

print("message_bytes = ", message_bytes)  # b'abc', 
# it should actually be the byte val but python tries to be helpful and shows the ascii representation of the bytes if they are printable
print("len(message_bytes) = ", len(message_bytes))  # 3

# slicing example
block = b"\x01\x02\x03\x04\x05\x06\x07\x08\x09\x0A" # this would be an immutable bytes object with 10 bytes
# slicing -> returns a 4-byte 'bytes' object
chunk = block[0:4]   # chunk = b'\x01\x02\x03\x04' (type: bytes)
print("chunk = ", chunk)


message_bytes =  b'abc'
len(message_bytes) =  3
chunk =  b'\x01\x02\x03\x04'


### 512-bit message blocks
Once the input message has been padded, sha256 processes it in 512bit chunks (64 bytes) AKA message blocks. Since the input msg is represented in `bytes`, naturally, each individual block would be a subslice of that big padded input message, meaning they would all be bytes that are 64-byte wide (let's ignore the metadata python attaches to the `bytes` object). And obviously, if there's more than 1 block, I'll be using a standard list ie `list[bytes]`

In [20]:
# initializing an individual block just for demo
block = bytes(range(64))

assert len(block) == 64
assert len(block) * 8 == 512

# in practice it'll be a direct consequence of the padded message, it'll look something like this, which we'll implement later
# I'll explain the padding later too, just know that it's always a multiple of 64 bytes
padded_msg = b"abc" + b"\x80" + b"\x00" * 52 + (24).to_bytes(8, "big")
assert len(padded_msg) == 64

blocks = [
    padded_msg[i : i + 64]
    for i in range(0, len(padded_msg), 64)
]
assert len(blocks) == 1
assert all(len(message_block) == 64 for message_block in blocks)

### the final 256-bit hash value
There's actually 2 useful representation for this, the final hash can be:
1. a 32 bytes for binary ops or storage
2. 64-char hexadecimal string for display (people usually display hashes in this because if we actually printed the 32 raw bytes, it'll be unreadable gibberish since many byte values won't correspond to printable ascii characters)

In [18]:
digest_bytes = bytes.fromhex(
    "ba7816bf8f01cfea414140de5dae2223"
    "b00361a396177a9cb410ff61f20015ad"
)

assert len(digest_bytes) == 32
assert len(digest_bytes) * 8 == 256

print()
print("the raw bytes = ", digest_bytes)  # b'\xba\x78\x16\xbf... you get the idea, but remember the python magic i talked about above, so it might show some ascii when printed
digest_hex = digest_bytes.hex()
print("the hex display = ", digest_hex)  # ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad


the raw bytes =  b'\xbax\x16\xbf\x8f\x01\xcf\xeaAA@\xde]\xae"#\xb0\x03a\xa3\x96\x17z\x9c\xb4\x10\xffa\xf2\x00\x15\xad'
the hex display =  ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad


### big-endian integers
big-endian just means it stores the most significant byte first, e.g. the integer 0x12345678 is stored as:
```
12 34 56 78
```
The endianness matters at three places in SHA-256:
1. at the input boundary: when we split the 64 byte message block (512bit) into 16 initial 32-bit words ($W_0 \dots W_{15}$). The FIPS standard says to interpret each 4-byte group as a big-endian 32-bit integer, so that's what we're following
2. during padding: the original message length in bits is appended as a 64-bit big-endian integer
3. at the output: when we convert the final 32-bit state variables ($H_0 \dots H_7$) back into the final 32-byte hash array, once again just following FIPS, we write out each state integer as 4 big-endian bytes.

I'll show the input and output boundaries in the code below. The padding example above demonstrates the big-endian length field.

NOTE:
inside the main loop, the values are represented as `np.uint32` and the bit shifts, rotations, `add32`, `Ch`, and `Maj` operations do not involve endianness. Endianness only applies when bytes are interpreted as integers or integers are written back into a byte sequence. The conversion helpers handle those byte boundaries.

In [19]:
value = 0x12345678

encoded = value.to_bytes(4, byteorder="big")
decoded = int.from_bytes(encoded, byteorder="big")

print(encoded)  # b'\x124Vx'
print(decoded == value)  # True

b'\x124Vx'
True


# implementation of data types
below is the setup for the data types used by future problems

In [5]:
import numpy as np

UINT32_MASK = 0xFFFFFFFF


def to_uint32(value: int | np.integer) -> np.uint32:
    """Convert an integer to a wrapped 32-bit unsigned NumPy value."""
    return np.uint32(int(value) & UINT32_MASK)


def to_uint32_array(values: list[int] | tuple[int, ...]) -> np.ndarray:
    """Convert a sequence of integers into a mutable uint32 NumPy array."""
    return np.array([to_uint32(value) for value in values], dtype=np.uint32)


def block_to_words(block: bytes) -> np.ndarray:
    """Decode one 64-byte, big-endian SHA-256 block into sixteen uint32 words."""
    if len(block) != 64:
        raise ValueError("a SHA-256 block must contain exactly 64 bytes")

    return np.frombuffer(block, dtype=">u4").astype(np.uint32)


def words_to_bytes(words: list[int] | tuple[int, ...] | np.ndarray) -> bytes:
    """Encode uint32 words as a big-endian byte sequence."""
    uint32_words = to_uint32_array(list(words))
    return uint32_words.astype(">u4").tobytes()


word = to_uint32(0x12345678)
words = to_uint32_array([0x12345678, 0x90ABCDEF, 0x00000001])
block = bytes(range(64))
block_words = block_to_words(block)

assert isinstance(word, np.uint32)
assert words.dtype == np.uint32
assert len(block_words) == 16
assert all(isinstance(value, np.uint32) for value in block_words)
assert words_to_bytes([0x12345678]) == bytes.fromhex("12345678")

print(f"word = {word:08x} ({type(word).__name__})")
print(f"words dtype = {words.dtype}")
print(f"block contains {len(block_words)} uint32 words")

word = 12345678 (uint32)
words dtype = uint32
block contains 16 uint32 words
